In [1]:
import rasterio as rio
import matplotlib.pyplot as plt
import os 

In [5]:
path = '/home/jovyan/work/AVOCA/GEM/Development/Anlaysis/TSA/monthly_lst/'

imfiles = [f for f in os.listdir(path) if f.startswith('GEMLST_monthly_') & f.endswith('.tif')]
print(len(imfiles))

312


In [ ]:
import numpy as np

with rio.open(path + imfiles[0]) as src:
    profile = src.profile
    agg = np.zeros(src.shape, dtype=np.float32)
    valid_count = np.zeros(src.shape, dtype=np.float32)

for imfile in imfiles:
    with rio.open(path + imfile) as src:
        data = src.read(1).astype(np.float32)
        valid = np.isfinite(data)
        agg[valid] += data[valid]
        valid_count[valid] += 1

avg_img = np.divide(
    agg,
    valid_count,
    out=np.full_like(agg, np.nan, dtype=np.float32),
    where=valid_count > 0,
)

In [ ]:
with rio.open(
    '/home/jovyan/work/AVOCA/GEM/Production/mean_lst_multidecade2.tiff',
    'w',
    driver='GTiff',
    height=avg_img.shape[0],
    width=avg_img.shape[1],
    count=1,
    dtype='float16',
    crs=profile['crs'],
    transform=profile['transform']
) as dst:
    dst.write(avg_img, 1)

print("Saved")

Saved
